# TTL feature ablation

**Question:** Sarhan et al. (2020) dropped `sttl`, `dttl` and `ct_state_ttl` from UNSW-NB15 because they correlate extremely strongly with the labels (likely testbed artifacts rather than real attack behaviour). Our pipeline keeps all three, and `ct_state_ttl` / `sttl` were among our top-ranked features. How much of our score depends on them?

**Method:** the best baseline (two-stage LightGBM: attack vs. Normal, then which attack) is trained twice, once with all features and once without the three TTL features. Trained on the original (not resampled) training data with balanced class weights, which the later graph-feature experiment showed to be better than the SMOTE data (macro F1 0.670 vs 0.653). Same hyper-parameters and held-out test set as the baseline notebook. Each setting is repeated with 3 seeds (a different validation split and model seed each time) so the effect of removing the features can be compared with ordinary run-to-run noise.

In [ ]:
import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

_fig_count = 0
_plt_show = plt.show


def show_and_save(*args, **kwargs):
    """Show every figure and also keep it as a PNG in the notebook output."""
    global _fig_count
    _fig_count += 1
    plt.savefig(f"/kaggle/working/fig_{_fig_count:02d}.png", dpi=130, bbox_inches="tight")
    _plt_show(*args, **kwargs)


plt.show = show_and_save

DATASET_PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
SEEDS = [42, 43, 44]
TTL_FEATURES = ["sttl", "dttl", "ct_state_ttl"]

## 1. Load data

Training uses `train.parquet` (original class balance, class weights instead of SMOTE) and evaluation uses `test.parquet`. Predictions of the all-features model are saved for the class-merge check.

In [ ]:
train_df = pd.read_parquet(f"{DATASET_PATH}/train.parquet")
test_df = pd.read_parquet(f"{DATASET_PATH}/test.parquet")
class_names = joblib.load(f"{DATASET_PATH}/preprocessing.pkl")["label_encoder"].classes_
n_classes = len(class_names)
normal_label = list(class_names).index("Normal")
attack_ids = [i for i in range(n_classes) if i != normal_label]
remap = {orig: new for new, orig in enumerate(attack_ids)}

all_features = [c for c in train_df.columns if c != "label"]
assert all(f in all_features for f in TTL_FEATURES)
feature_sets = {
    "all features": all_features,
    "without TTL features": [c for c in all_features if c not in TTL_FEATURES],
}
print({name: len(cols) for name, cols in feature_sets.items()}, "| test rows:", len(test_df))

## 2. Two-stage LightGBM

Same as the baseline: Stage 1 separates attack from Normal, Stage 2 (trained on attack rows only) picks the attack type. Early stopping uses a validation slice of the training data; the test set is only used at the end.

In [ ]:
def run_two_stage(features, seed):
    X_tr, X_val, y_tr, y_val = train_test_split(
        train_df[features], train_df["label"], test_size=0.15, stratify=train_df["label"], random_state=seed
    )
    params = dict(class_weight="balanced", learning_rate=0.08, num_leaves=63,
                  n_estimators=300, random_state=seed, verbose=-1)
    stop = [lgb.early_stopping(20), lgb.log_evaluation(0)]

    stage1 = lgb.LGBMClassifier(objective="binary", **params)
    stage1.fit(X_tr, (y_tr != normal_label).astype(int),
               eval_set=[(X_val, (y_val != normal_label).astype(int))], callbacks=stop)

    a_tr, a_val = y_tr != normal_label, y_val != normal_label
    stage2 = lgb.LGBMClassifier(objective="multiclass", num_class=len(attack_ids), **params)
    stage2.fit(X_tr[a_tr], y_tr[a_tr].map(remap),
               eval_set=[(X_val[a_val], y_val[a_val].map(remap))], callbacks=stop)

    X_test = test_df[features]
    pred = np.full(len(X_test), normal_label)
    flagged = np.where(stage1.predict(X_test) == 1)[0]
    pred[flagged] = np.array(attack_ids)[stage2.predict(X_test.iloc[flagged])]
    return pred


y_test = test_df["label"].to_numpy()
rows, per_class, saved_preds = [], [], {}
for name, features in feature_sets.items():
    for seed in SEEDS:
        pred = run_two_stage(features, seed)
        if name == "all features":
            saved_preds[f"pred_seed{seed}"] = pred
        f1s = f1_score(y_test, pred, labels=np.arange(n_classes), average=None, zero_division=0)
        rows.append({"features": name, "seed": seed, "macro F1": f1s.mean(),
                     "macro F1 (attacks only)": np.delete(f1s, normal_label).mean()})
        per_class.append(pd.Series(f1s, index=class_names).rename((name, seed)))
        print(f"{name:<22} seed {seed} | macro F1 {rows[-1]['macro F1']:.4f} | attacks only {rows[-1]['macro F1 (attacks only)']:.4f}")

pd.DataFrame(saved_preds, index=test_df.index).to_parquet("/kaggle/working/final_model_test_pred.parquet")

## 3. Results

Bars = mean over the 3 seeds, error bars = one standard deviation, dots = individual runs. If the two bars overlap within the error bars, removing the TTL features did not change the score by more than normal run-to-run variation.

In [ ]:
res = pd.DataFrame(rows)
summary = res.groupby("features")[["macro F1", "macro F1 (attacks only)"]].agg(["mean", "std"]).round(4)
print(summary.to_string())

long = res.melt(id_vars=["features", "seed"], value_vars=["macro F1", "macro F1 (attacks only)"],
                var_name="metric", value_name="score")
plt.figure(figsize=(8, 4.5))
sns.barplot(data=long, x="metric", y="score", hue="features", errorbar="sd", capsize=0.1, alpha=0.8)
sns.stripplot(data=long, x="metric", y="score", hue="features", dodge=True, palette=["black", "black"], size=5, legend=False)
plt.ylabel("test macro F1")
plt.xlabel("")
plt.title("Two-stage LightGBM: with vs without the TTL features (3 seeds each)")
plt.tight_layout()
plt.show()

In [ ]:
pc = pd.DataFrame(per_class)
pc.index = pd.MultiIndex.from_tuples(pc.index, names=["features", "seed"])
mean_pc = pc.groupby(level="features").mean().T
std_pc = pc.groupby(level="features").std().T
mean_pc.plot.bar(figsize=(11, 4.5), yerr=std_pc, capsize=2, width=0.8)
plt.ylabel("test F1")
plt.title("Per-class F1 (mean of 3 seeds, error bars = std)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
print(mean_pc.assign(change=mean_pc["without TTL features"] - mean_pc["all features"]).round(3).to_string())